# NumPy dtypes, Copy & View

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 3/7

Two ideas separate NumPy beginners from confident users: every array has a FIXED numeric type with real limits, and arrays can share memory without you noticing. This lesson makes both explicit so they never surprise you again.

## 🎯 Learning Objectives

- Identify the common NumPy dtypes and what each is used for
- Predict (and avoid) silent integer overflow with small int types
- Convert arrays safely with `astype` and the `dtype=` argument
- Distinguish `b = a`, `.view()` and `.copy()` using `.base` and mutation tests
- State which operations return views and which return copies
- Check shared memory deliberately with `np.shares_memory`

## 1. Every Array Has One dtype

Unlike Python lists, a NumPy array holds elements of ONE type, declared up front. That homogeneity is exactly why operations are fast — but it means types have fixed sizes and limits.

| Family | Dtypes | Size / Range |
|---|---|---|
| Signed integers | `int8`, `int16`, `int32`, `int64` | 1–8 bytes; `int8` holds −128..127 |
| Unsigned integers | `uint8`, `uint16`, `uint32`, `uint64` | 0 and up; `uint8` = 0..255 (image pixels!) |
| Floats | `float16`, `float32`, `float64` | half / single / double precision; default is `float64` |
| Boolean | `bool` | 1 byte per value; the language of masks |
| Complex | `complex64`, `complex128` | two floats: real + imaginary part |
| Strings | `<U5`, `<U10`, ... | fixed-width Unicode; number = max characters |

**Syntax:**
```python
arr.dtype                     # inspect
np.zeros(3, dtype=np.float32) # request at creation
arr.astype(np.int64)          # convert afterwards
```

Handy reference objects tell you each type's exact limits:

In [ ]:
import numpy as np

print("int8   :", np.iinfo(np.int8))
print()
print("uint8  :", np.iinfo(np.uint8))
print()
print("float32:", np.finfo(np.float32))

## 2. ⚠️ Overflow: When Numbers Outgrow Their Type

A fixed-size type cannot grow like a Python int. When a value exceeds its limit, NumPy **wraps around silently** — no error, just a wrong-looking answer. This bites image code (`uint8`) constantly.

**Syntax:**
```python
tiny = np.array([126, 127], dtype=np.int8)
tiny + 1        # wraps: 127 + 1 becomes -128 !
```

In [ ]:
import numpy as np

tiny = np.array([100, 126, 127], dtype=np.int8)   # int8 max is 127
print(tiny + 1)      # [101 -127 -128] - wrapped around, no warning!

In [ ]:
import numpy as np

pixels = np.array([200, 240, 250], dtype=np.uint8)  # uint8 max is 255
brightened = pixels + 60                            # naive brightening...
print(brightened)                                   # ...wraps past 255 back to tiny values!

safe = pixels.astype(np.int16) + 60                 # widen FIRST, then do math
print(safe)

## 3. Converting Types with astype

`astype` returns a NEW array with the requested dtype — the original is untouched. Converting float to int TRUNCATES toward zero rather than rounding, so round explicitly when that matters.

**Syntax:**
```python
arr.astype(int)              # to platform int (usually int64)
arr.astype(np.float32)       # shrink precision - common before ML training
np.round(arr).astype(int)    # proper rounding, then cast
```

In [ ]:
import numpy as np

prices = np.array([19.99, 5.49, 8.75])

print("prices :", prices)
print("astype :", prices.astype(int))            # truncates toward ZERO
print("rounded:", np.round(prices).astype(int))  # round first, then cast

downcast = prices.astype(np.float32)             # half the memory of float64
print("float32:", downcast.dtype, downcast)

In [ ]:
import numpy as np

raw = np.array(["42", "17", "93"])     # numbers read from a text file / CSV
nums = raw.astype(int)
print(nums + 8)                        # now arithmetic works

flags = np.array([1, 0, 1, 1]).astype(bool)   # ints -> boolean mask
print(flags)

## 4. Choosing the dtype at Creation

NumPy guesses sensibly (ints become `int64`, decimals become `float64`), but you can — and often should — declare the type yourself at creation time.

**Syntax:**
```python
np.array([1, 2, 3], dtype=np.float32)
np.zeros(5, dtype=bool)
```

In [ ]:
import numpy as np

a = np.array([1, 2, 3])                    # guessed: int64
b = np.array([1.0, 2.5])                   # guessed: float64
c = np.array([1, 2, 3], dtype=np.float32)  # forced: float32
d = np.array([1, 0, 1], dtype=bool)        # forced: bool

print(a.dtype, b.dtype, c.dtype, d.dtype)
print(d)

## 5. Assignment Is Not Copying: b = a

The first surprise: `b = a` creates NO new array. It just gives the same object a second name. Every edit through either name is visible through both.

**Syntax:**
```python
b = a          # second NAME, same single object
a is b         # True - literally the same object
```

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
b = a                        # NO copy - just another name

print("same object?", a is b)
b[0] = 99                    # edit via b...
print("a after editing b:", a)   # ...and a changed too

## 6. view(): New Wrapper, Same Data

`.view()` builds a genuinely NEW array object — but one that points at the SAME underlying data buffer. Different identity, shared bytes.

**Syntax:**
```python
v = a.view()               # new object, same data memory
v.base is a                # views know who owns their data
np.shares_memory(a, v)     # True
```

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
v = a.view()

print("different objects? ", v is not a)
print("shares memory?    ", np.shares_memory(a, v))
print("v.base is a?      ", v.base is a)   # .base points to the data owner

v[1] = 500
print("a after editing v:", a)             # data is shared, so a changed

> 🔍 **Under the Hood:** A NumPy array is really two things: a small header (dtype, shape, strides, refcount) plus a pointer to a data buffer. `b = a` shares the HEADER itself — both names are one object, hence `a is b`. `.view()` mints a fresh header aimed at the SAME buffer, which is why you can even give a view a different shape and reinterpret identical bytes. `.copy()` allocates a second buffer and memcpy's the data across. The `.base` attribute exposes who owns the bytes: views have it set, true copies have `base is None`. This is why slicing, reshaping and transposing are nearly free — they only touch headers.

## 7. copy(): Fully Independent

`.copy()` allocates brand-new storage and duplicates the values. Afterwards the arrays share nothing.

**Syntax:**
```python
c = a.copy()               # independent buffer
np.shares_memory(a, c)     # False
c.base is None             # True - c owns its own data
```

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
c = a.copy()

print("shares memory?", np.shares_memory(a, c))
c[0] = 777
print("a unchanged :", a)
print("c modified  :", c)

## 8. View or Copy? What NumPy Returns

The rule that prevents most surprise-bug reports:

| Operation | Returns | Why |
|---|---|---|
| Basic slicing `a[1:4]`, `a[:, 0]` | **View** | a rectangular patch of existing memory |
| `reshape` / `ravel` (when possible) | **View** | same bytes, new shape metadata |
| Transpose `.T` | **View** | only strides change |
| `a.view()` | **View** | explicitly requested |
| Fancy indexing `a[[0, 2]]` | **Copy** | gathers scattered elements into a new buffer |
| Boolean mask `a[a > 5]` | **Copy** | builds a fresh result array |
| Arithmetic `a + 1` | **Copy** | always produces a new array |
| `a.copy()` / `np.copy(a)` | **Copy** | explicitly requested |

The intuition: a view is only possible when the answer is "a regular patch of existing memory". Masks and fancy indices jump around arbitrarily, so NumPy must assemble a new buffer.

In [ ]:
import numpy as np

a = np.arange(6)

candidates = [("slice  a[1:5]", a[1:5]),
              ("fancy  a[[1,2,3]]", a[[1, 2, 3]]),
              ("mask   a[a>2]", a[a > 2]),
              ("arith  a+100", a + 100)]

for label, candidate in candidates:
    print(label, "-> shares memory:", np.shares_memory(a, candidate))

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Using `b = a` as a backup | It is the same object — nothing was saved | Use `b = a.copy()` |
| Editing a masked selection and expecting changes upstream | `kept = a[a > 0]; kept[:] = 0` edits only the COPY | Apply the mask directly: `a[a > 0] = 0` |
| Trusting `astype(int)` to round | It truncates toward zero: `19.99` → `19` | `np.round(arr).astype(int)` |
| Doing arithmetic on `uint8` pixels | Values wrap around 255 silently | Widen first: `arr.astype(np.int16)` or `float` |
| Comparing floats with `==` after converting to `float32` | Precision loss breaks equality tests | Use `np.isclose(a, b)` / `np.allclose(a, b)` |

## 💡 Best Practices & Pro Tips

- Stay on the defaults (`int64` / `float64`) unless you have a reason; drop to `float32` when arrays get large or go to a GPU.
- Call `.copy()` the moment you *mean* independent data — treat it as documentation of intent.
- When an array mutates mysteriously, interrogate lineage with `arr.base` and `np.shares_memory(a, b)`.
- Images are `uint8` (0–255): cast to float, process, then cast back — never do arithmetic in uint8.
- **AI-engineering relevance:** dtype choices dominate GPU memory budgets (fp16/bf16 training), and view mechanics are why `reshape().transpose()` pipelines process gigabytes without doubling RAM.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `arr.dtype` | Element type of the array | `a.dtype` → `int64` |
| `int8..int64, uint8..` | Integer families with fixed ranges | `np.iinfo(np.int8)` |
| `float16/32/64` | Half / single / double precision floats | `np.finfo(np.float32)` |
| `bool`, `complex64/128`, `<U5` | Flags, complex numbers, strings | `np.ones(3, dtype=bool)` |
| `arr.astype(t)` | Convert to a NEW typed copy | `prices.astype(int)` |
| `np.array(..., dtype=)` | Fix the type at creation | `np.array([1, 2], dtype=np.float32)` |
| `b = a` | Second name, same object | `a is b` → `True` |
| `a.view()` | New wrapper, shared data | `v.base is a` → `True` |
| `a.copy()` | Independent buffer | `c.base is None` → `True` |
| `np.shares_memory(a, b)` | Test whether data overlaps | debugging tool |

Key takeaways:
- One dtype per array — powerful for speed, dangerous at the edges of its range.
- Integer overflow wraps silently; widen the type before arithmetic near limits.
- Assignment copies nothing; `.view()` shares data; only `.copy()` is truly independent.
- Basic slicing yields views; masks, fancy indexing and arithmetic yield copies.

## 🔗 Next Lesson

- Continue to **[04_Shape_Reshape_Broadcasting](../04_Shape_Reshape_Broadcasting/notes.ipynb)** — reshaping, stacking, and the broadcasting rules that make mismatched shapes work.